# ⚡ PowerGrid EV Analytics — Task 01: Demand Forecasting
**Author:** Adeel Anwar  
**Objective:** End-to-End Time-Series Analysis & SARIMA Demand Forecasting for EV Charging Networks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully!")

## 1. Load Processed Time-Series Data

In [ ]:
df_daily = pd.read_csv('../data/processed/daily_demand_total.csv', parse_dates=['timestamp'])
df_daily.set_index('timestamp', inplace=True)
df_daily = df_daily.asfreq('D')
df_daily.head()

## 2. Visualize Historical Daily Demand

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_daily.index, df_daily['total_energy_kwh'], color='#1f77b4', lw=1.5)
plt.title('Daily Total Energy Demand (2022 - 2024)', fontsize=14, weight='bold')
plt.ylabel('Energy Demand (kWh)')
plt.xlabel('Date')
plt.show()

## 3. Stationarity Testing (ADF & KPSS)

In [ ]:
adf_raw = adfuller(df_daily['total_energy_kwh'].dropna())
print(f"Raw Series ADF Statistic: {adf_raw[0]:.4f} (p-value: {adf_raw[1]:.4f})")

diff_1 = df_daily['total_energy_kwh'].diff().dropna()
adf_diff = adfuller(diff_1)
print(f"First Difference ADF Statistic: {adf_diff[0]:.4f} (p-value: {adf_diff[1]:.4e})")

## 4. Seasonal Decomposition (Weekly Cycle s=7)

In [ ]:
decomp = seasonal_decompose(df_daily['total_energy_kwh'], model='additive', period=7)
fig = decomp.plot()
fig.set_size_inches(14, 9)
plt.show()

## 5. Fit SARIMA Model & Evaluate on 90-Day Test Set

In [ ]:
test_days = 90
train = df_daily['total_energy_kwh'].iloc[:-test_days]
test = df_daily['total_energy_kwh'].iloc[-test_days:]

model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7), enforce_stationarity=False, enforce_invertibility=False)
results = model.fit(disp=False)

preds = results.get_forecast(steps=len(test)).predicted_mean
mape = np.mean(np.abs((test - preds) / test)) * 100
mae = mean_absolute_error(test, preds)
rmse = np.sqrt(mean_squared_error(test, preds))
r2 = r2_score(test, preds)

print(f"Test Set Evaluation:")
print(f"  • MAPE : {mape:.2f}%")
print(f"  • MAE  : {mae:,.2f} kWh")
print(f"  • RMSE : {rmse:,.2f} kWh")
print(f"  • R²   : {r2:.4f}")

## 6. Forward Forecast into Q1 2025

In [ ]:
full_model = SARIMAX(df_daily['total_energy_kwh'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 7), enforce_stationarity=False, enforce_invertibility=False)
full_res = full_model.fit(disp=False)
future_forecast = full_res.get_forecast(steps=90)

plt.figure(figsize=(15, 6))
plt.plot(df_daily.index[-365:], df_daily['total_energy_kwh'].iloc[-365:], label='2024 Historical Demand', color='#1f77b4')
plt.plot(future_forecast.predicted_mean.index, future_forecast.predicted_mean, label='Q1 2025 Forecast', color='#ff7f0e', lw=2)
plt.fill_between(future_forecast.predicted_mean.index, future_forecast.conf_int().iloc[:, 0], future_forecast.conf_int().iloc[:, 1], color='#ff7f0e', alpha=0.2, label='95% CI')
plt.title('PowerGrid EV Demand Forecast — Q1 2025', fontsize=14, weight='bold')
plt.ylabel('Daily Energy (kWh)')
plt.legend()
plt.show()